# 09 — Nano mobile : Luna + tes vidéos, à 960 px

Cet essai entraîne **YOLO26n à 960 px**, avec 1 547 images de train et 168 de validation. On repart des poids COCO : le modèle précédent sert de référence, pas d'initialisation.

Le calcul tourne sur **Hugging Face A10G**, pas sur le Mac. Ce notebook présente la configuration, le suivi et les résultats ; **aucune cellule ne lance un nouveau job payant**.

Les 292 images du test personnel ne servent ni à l'entraînement ni au choix du meilleur checkpoint. La validation reste séparée par groupes de matchs de la compilation Luna.

In [ ]:
from pathlib import Path
import csv
import json
import subprocess
import sys
from IPython.display import display, Markdown, Image

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "configs/mobile_nano_960.json").is_file())
CONFIG_PATH = PROJECT_ROOT / "configs/mobile_nano_960.json"
CONFIG = json.loads(CONFIG_PATH.read_text())
JOB_PATH = PROJECT_ROOT / "models/reports" / (CONFIG["experiment_id"] + "_job.json")
JOB = json.loads(JOB_PATH.read_text()) if JOB_PATH.exists() else None
ACTUAL_CONFIG = JOB["config"] if JOB else CONFIG
RUN_ROOT = PROJECT_ROOT / "models/checkpoints/baseline_runs" / ACTUAL_CONFIG["experiment_id"]
ARTIFACT_ROOT = RUN_ROOT / "artifacts"
print("Projet :", PROJECT_ROOT)
print("Résultats :", ARTIFACT_ROOT)

## 1. Quelles données apprend-il ?

On mélange la première compilation revue et les trois clips personnels de train. On ne réintroduit pas l'ancien Infactory TV dans ce premier essai.

**37 négatifs ambigus** du train/validation (absence de réponse Luna suivie d'un label vide) sont mis en quarantaine. Les images corrigées en positif restent présentes. Aucun dataset parent ni aucune décision humaine n'est modifié.

Les JPEG locaux sont des liens physiques : pas de seconde copie permanente des images. La validation contient deux groupes de matchs identifiés visuellement : elle reste petite et ne représente pas toutes les situations personnelles.

In [ ]:
DATA = json.loads((PROJECT_ROOT / ACTUAL_CONFIG["dataset"] / "build_report.json").read_text())
lines = ["| Split | Images | Avec ballon | Sans ballon |", "|---|---:|---:|---:|"]
for split, counts in DATA["splits"].items():
    lines.append(f"| {split} | {counts['images']} | {counts['positive']} | {counts['negative']} |")
display(Markdown("\n".join(lines)))
print("Quarantaine :", DATA["quarantined_empty_missing_luna"])
print("Images de test incluses :", DATA["test_images_included"])
print("Doublons exacts :", DATA["exact_image_duplicates"])
print("Chevauchement train/val par match :", DATA["train_val_match_overlap"])

## 2. Configuration réellement lancée

60 époques maximum, arrêt après 12 époques sans progrès de validation, batch 8, images 960 px. Le GPU a une limite dure d'une heure. Les checkpoints sont sauvegardés sur HF toutes les cinq époques.

`best.pt` est choisi par la validation, **jamais par le test**. Les poids et les images de cet essai restent privés. Une modification du fichier de configuration ne change pas le job déjà lancé ; la cellule affiche son instantané.

In [ ]:
display(Markdown("\n".join([
    f"- Modèle : **{ACTUAL_CONFIG['base_model']}**",
    f"- Résolution : **{ACTUAL_CONFIG['imgsz']} px**",
    f"- Époques maximum : **{ACTUAL_CONFIG['epochs']}** ; patience : **{ACTUAL_CONFIG['patience']}**",
    f"- Batch : **{ACTUAL_CONFIG['batch']}** ; seed : **{ACTUAL_CONFIG['seed']}**",
    f"- GPU : **{ACTUAL_CONFIG['hardware']}** ; limite : **{ACTUAL_CONFIG['job_timeout']}**",
])))
if JOB:
    display(Markdown(f"[Ouvrir le job Hugging Face]({JOB['job_url']})"))
else:
    print("Le job n'est pas encore enregistré. Aucun lancement depuis cette cellule.")

## 3. Suivre et récupérer

Réexécuter cette cellule fait uniquement une consultation de statut et, si le job est terminé, le téléchargement des résultats. Elle ne crée ni ne relance de job.

Tant qu'il tourne, les résultats intermédiaires restent accessibles dans le dépôt privé HF. Si le job échoue ou atteint sa limite, on ne présente pas un checkpoint intermédiaire comme un entraînement terminé.

In [ ]:
from huggingface_hub import HfApi
if JOB_PATH.exists():
    JOB = json.loads(JOB_PATH.read_text())
    info = HfApi().inspect_job(job_id=JOB["job_id"])
    print("État :", info.status.stage)
    if info.status.message:
        print(info.status.message)
    if info.status.stage == "COMPLETED":
        subprocess.run([sys.executable, str(PROJECT_ROOT / "scripts/mobile_training_hf.py"),
                        "collect", "--config", str(CONFIG_PATH)],
                       cwd=PROJECT_ROOT, check=True)
    else:
        print("Les graphiques finaux apparaîtront après la collecte.")
else:
    print("Aucun job enregistré pour cet essai.")

## 4. Comparaison à résolution égale

Trois modèles évalués sur les **mêmes 168 images à 960 px** : COCO générique, précédent nano entraîné sur Luna, nouveau nano Luna + tes vidéos.

Précision, rappel, F1, TP, FP, FN : **confiance ≥ 0,25 et IoU ≥ 0,50**. Les mAP intègrent différents seuils de confiance : elles ne correspondent pas à ce seul point de fonctionnement.

C'est une comparaison d'expériences (nouvelles données **et** entraînement différent), pas une preuve isolant l'effet de chaque nouvelle image. Les temps GPU HF ne prédisent pas la vitesse ni la batterie sur iPhone.

In [ ]:
REPORT_PATH = ARTIFACT_ROOT / "training_report.json"
REPORT = json.loads(REPORT_PATH.read_text()) if REPORT_PATH.exists() else None
if REPORT and REPORT.get("status") == "complete":
    results = REPORT["validation"]
    lines = ["| Modèle @960 | mAP50 | mAP50–95 | Précision | Rappel | F1 | TP | FP | FN | FP négatifs |",
             "|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|"]
    for key, label in [("pretrained", "COCO"), ("previous_luna", "Ancien nano Luna"), ("mobile", "Nouveau nano")]:
        result = results[key]
        fixed = result["fixed"]
        numbers = [result["map50"], result["map50_95"], fixed["precision"], fixed["recall"], fixed["f1"]]
        lines.append("| " + label + " | " + " | ".join(f"{x:.1%}" for x in numbers)
                     + " | " + " | ".join(str(fixed.get(k, 0)) for k in ["tp", "fp", "fn", "fp_on_negatives"]) + " |")
    display(Markdown("\n".join(lines)))
    before, after = results["previous_luna"], results["mobile"]
    display(Markdown(f"Écart de mAP50 : **{100 * (after['map50'] - before['map50']):+.1f} points** ; "
                     f"écart de F1 : **{100 * (after['fixed']['f1'] - before['fixed']['f1']):+.1f} points**."))
    print("Époques réalisées :", REPORT["epochs_logged"])
    print("Temps d'entraînement :", round(REPORT["training_seconds"] / 60, 1), "minutes")
else:
    print("Rapport final indisponible : entraînement ou collecte en cours.")

## 5. Loss et validation au fil des époques

On cherche une baisse des pertes et une amélioration des métriques de validation. Une loss train qui baisse pendant que la validation se dégrade durablement suggère du surapprentissage ; une seule oscillation ne suffit pas.

Les courbes précision/rappel de `results.csv` viennent du calcul Ultralytics : leur point de fonctionnement peut différer du seuil fixe 0,25 du tableau.

In [ ]:
import matplotlib.pyplot as plt
from io import BytesIO

def show_figure(fig):
    with BytesIO() as buffer:
        fig.savefig(buffer, format="png", dpi=120, bbox_inches="tight")
        display(Image(data=buffer.getvalue()))
    plt.close(fig)
HISTORY_PATH = ARTIFACT_ROOT / "train/results.csv"
if HISTORY_PATH.exists():
    with HISTORY_PATH.open() as stream:
        history = [{k.strip(): float(v) for k, v in row.items()} for row in csv.DictReader(stream)]
    epochs = [row["epoch"] for row in history]
    loss_names = [name for name in ["box_loss", "cls_loss", "l1_loss", "dfl_loss"]
                  if "train/" + name in history[0] and "val/" + name in history[0]]
    fig, axes = plt.subplots(1, len(loss_names), figsize=(5 * len(loss_names), 3.6), squeeze=False)
    for axis, name in zip(axes[0], loss_names):
        for split in ["train", "val"]:
            axis.plot(epochs, [row[split + "/" + name] for row in history], label=split)
        axis.set(title=name, xlabel="Époque")
        axis.legend()
        axis.grid(alpha=.2)
    fig.tight_layout()
    show_figure(fig)
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
    for axis, metrics in zip(axes, [("metrics/mAP50(B)", "metrics/mAP50-95(B)"),
                                  ("metrics/precision(B)", "metrics/recall(B)")]):
        for metric in metrics:
            if metric in history[0]:
                axis.plot(epochs, [row[metric] for row in history], label=metric)
        axis.set(xlabel="Époque", ylim=(0, 1))
        axis.legend()
        axis.grid(alpha=.2)
    fig.tight_layout()
    show_figure(fig)
else:
    print("Historique disponible après collecte.")

## 6. Matrices de confusion au même seuil

Colonnes = annotation ; lignes = prédiction. En haut à gauche : bonnes détections (TP). En haut à droite : fausses détections (FP). En bas à gauche : ballons manqués (FN).

La case du bas à droite reste vide : en détection d'objets, on ne compte pas toutes les régions de fond comme des vrais négatifs. Ces matrices personnalisées utilisent exactement les seuils du tableau, contrairement aux matrices standard dont les seuils peuvent différer.

In [ ]:
import base64
from IPython.display import HTML
panels = []
for key, label in [("pretrained", "COCO"), ("previous_luna", "Ancien nano Luna"), ("mobile", "Nouveau nano")]:
    path = ARTIFACT_ROOT / "evaluation" / key / "fixed_confusion_matrix.png"
    if path.exists():
        encoded = base64.b64encode(path.read_bytes()).decode()
        panels.append(f'<div style="flex:1;min-width:230px"><p>{label}</p><img style="width:100%" src="data:image/png;base64,{encoded}"></div>')
if panels:
    display(HTML('<div style="display:flex;flex-wrap:wrap;gap:12px">' + "".join(panels) + "</div>"))
else:
    print("Matrices disponibles après collecte.")

## 7. Courbes précision–rappel et choix du seuil

La courbe PR montre le compromis précision/rappel. La courbe F1 indique les seuils qui équilibrent les deux sur **la validation**. On ne choisit pas un seuil en optimisant sur le test personnel.

In [ ]:
evaluation = ARTIFACT_ROOT / "evaluation/mobile"
for pattern in ["*PR_curve.png", "*F1_curve.png"]:
    paths = sorted(evaluation.glob(pattern))
    if paths:
        display(Image(filename=str(paths[0]), width=650))
    else:
        print("Courbe absente pour le moment :", pattern)

## 8. Test personnel final — ancien et nouveau à 960 px

Le modèle et les seuils ont été figés avant de consulter ce test. Les 292 images revues comprennent 264 positives et 28 négatives, avec 279 boîtes (11 images multiboîtes). Les 88 exclusions ne sont pas évaluées.

On présente séparément la détection stricte (IoU ≥ 0,50, toutes les boîtes) et le choix d'une seule cible (maximum de confiance, IoU ≥ 0,10 ou 0,20). Avec plusieurs annotations, toute boîte annotée peut correspondre : ce test n'identifie pas avec certitude le ballon en jeu.

**Limite : le train long et le test court partagent le match du 30 septembre.** Ces images ne sont pas une validation supplémentaire et ne servent pas à choisir de nouvelles époques ou un seuil optimal.


In [ ]:
PERSONAL_ROOT = PROJECT_ROOT / "models/reports/mobile_personal_test_20261010"
PERSONAL_PATH = PERSONAL_ROOT / "report.json"
PERSONAL = json.loads(PERSONAL_PATH.read_text()) if PERSONAL_PATH.exists() else None
if PERSONAL and PERSONAL.get("metrics_status") == "complete":
    results = PERSONAL["detection"]
    lines = ["| Test personnel @960 | Ancien nano | Nouveau nano |",
             "|---|---:|---:|"]
    before, after = results["previous_luna"], results["mobile"]
    for label, key in [("mAP50", "map50"), ("mAP50–95", "map50_95")]:
        lines.append(f"| {label} | {before[key]:.1%} | {after[key]:.1%} |")
    for label, key in [("Précision", "precision"), ("Rappel", "recall"), ("F1", "f1")]:
        lines.append(f"| {label} | {before['fixed'][key]:.1%} | {after['fixed'][key]:.1%} |")
    for label, key in [("Bonnes détections", "tp"), ("Fausses détections", "fp"),
                       ("Ballons ratés", "fn"), ("FP sur négatifs", "fp_on_negatives")]:
        lines.append(f"| {label} | {before['fixed'].get(key, 0)} | {after['fixed'].get(key, 0)} |")
    display(Markdown("\n".join(lines)))
    lines = ["| Cadrage : une seule cible, conf ≥ 0,25 | Ancien nano | Nouveau nano |",
             "|---|---:|---:|"]
    for overlap in ["0.1", "0.2"]:
        before = PERSONAL["targeting"][overlap]["previous_luna"]["metrics"]
        after = PERSONAL["targeting"][overlap]["mobile"]["metrics"]
        lines.append(f"| Bonnes décisions, IoU ≥ {overlap} | {before['correct_decision_rate']:.1%} | {after['correct_decision_rate']:.1%} |")
    display(Markdown("\n".join(lines)))
    for key in ["previous_luna", "mobile"]:
        path = PERSONAL_ROOT / key / "fixed_confusion_matrix.png"
        if path.exists():
            display(Image(filename=str(path), width=450))
else:
    print("Évaluation personnelle en cours ou pas encore exécutée.")

## 9. Revoir la courte vidéo

Le nouveau nano analyse **toutes les frames du clip**, pas seulement celles annotées. Une seule boîte est affichée : la plus confiante au-dessus de 0,25. Le cadre vert est légèrement élargi pour être visible ; les coordonnées évaluées restent les coordonnées originales.

La confiance est dans une bande au-dessus de l'image, pas dans la petite boîte. Il n'y a ni tracker ni lissage temporel : des sauts de cible et des disparitions restent possibles. Des frames exclues du benchmark apparaissent dans la vidéo complète, sans être ajoutées aux métriques.

L'inférence est séquentielle sur CPU, sans nouveau calcul HF payant. La vidéo reste un fichier séparé pour ne pas gonfler le notebook.

In [ ]:
from IPython.display import Video
if PERSONAL_PATH.exists():
    PERSONAL = json.loads(PERSONAL_PATH.read_text())
if PERSONAL and PERSONAL.get("status") == "complete":
    VIDEO_PATH = Path(PERSONAL["video"]["path"])
    print("Vidéo :", VIDEO_PATH)
    print("Durée :", round(PERSONAL["video"]["duration_seconds"], 1), "s")
    display(Video(filename=str(VIDEO_PATH), embed=False, width=960))
else:
    print("La vidéo sera disponible une fois l'inférence terminée.")

## 10. Essai contrôlé : ajouter les anciennes images TV

Nous ajoutons **3 048 images du train v002** aux **1 547 images récentes**, sans changer les **168 images de validation**. Aucun ancien val/test n'est ajouté au train. YOLO26n reste à 960 px et repart du même poids COCO ; ce n'est pas une continuation du premier modèle.

Même recette, batch et maximum d'époques ; mais davantage de mises à jour par époque et de négatifs. Ce test mesure l'intérêt de cet ajout précis, pas uniquement la qualité des annotations. Le benchmark personnel a déjà été inspecté et n'est pas un test indépendant par match.

La cellule ci-dessous lit seulement les fichiers locaux : aucun lancement de GPU ni nouvelle inférence. Après la fin du job, récupérer ses résultats avec `python scripts/mobile_training_hf.py collect --config configs/mobile_nano_tv_960.json`, puis réexécuter cette cellule.

In [ ]:
TV_CONFIG = json.loads((PROJECT_ROOT / 'configs/mobile_nano_tv_960.json').read_text())
TV_DATA = json.loads((PROJECT_ROOT / TV_CONFIG['dataset'] / 'build_report.json').read_text())
assert TV_DATA['validation_identical_to_baseline'] and TV_DATA['test_images_included'] == 0
display(Markdown('| Données | Récent seul | Récent + TV |\n|---|---:|---:|\n| Train | 1 547 | 4 595 |\n| Validation identique | 168 | 168 |'))
TV_JOB_PATH = PROJECT_ROOT / 'models/reports' / (TV_CONFIG['experiment_id'] + '_job.json')
if TV_JOB_PATH.exists():
    TV_JOB = json.loads(TV_JOB_PATH.read_text())
    print('Dernier état enregistré (pas un suivi en direct) :', TV_JOB['status'])
    if TV_JOB.get('job_url'):
        display(Markdown('[Ouvrir le job HF](' + TV_JOB['job_url'] + ')'))
TV_ARTIFACTS = PROJECT_ROOT / 'models/checkpoints/baseline_runs' / TV_CONFIG['experiment_id'] / 'artifacts'
TV_REPORT_PATH = TV_ARTIFACTS / 'training_report.json'
if TV_REPORT_PATH.exists():
    TV_REPORT = json.loads(TV_REPORT_PATH.read_text())
    if TV_REPORT['status'] == 'complete':
        before, after = TV_REPORT['validation']['modern_only'], TV_REPORT['validation']['mobile']
        lines = ['| Même validation @960 | Récent seul | Récent + TV | Écart |', '|---|---:|---:|---:|']
        for title, key in [('mAP50', 'map50'), ('mAP50–95', 'map50_95')]:
            lines.append(f"| {title} | {before[key]:.1%} | {after[key]:.1%} | {(after[key]-before[key])*100:+.1f} pts |")
        for title, key in [('Précision', 'precision'), ('Rappel', 'recall'), ('F1', 'f1')]:
            b, a = before['fixed'][key], after['fixed'][key]
            lines.append(f'| {title} | {b:.1%} | {a:.1%} | {(a-b)*100:+.1f} pts |')
        for title, key in [('Bonnes détections', 'tp'), ('Fausses détections', 'fp'), ('Ballons ratés', 'fn'), ('FP sur négatifs', 'fp_on_negatives')]:
            b, a = before['fixed'][key], after['fixed'][key]
            lines.append(f'| {title} | {b} | {a} | {a-b:+d} |')
        display(Markdown('\n'.join(lines)))
        for name in ['modern_only', 'mobile']:
            image_path = TV_ARTIFACTS / 'evaluation' / name / 'fixed_confusion_matrix.png'
            if image_path.exists():
                display(Image(filename=str(image_path), width=450))
        learning_plot = TV_ARTIFACTS / 'train/results.png'
        if learning_plot.exists():
            display(Image(filename=str(learning_plot), width=1000))
    else:
        print('Résultats intermédiaires : pas encore de conclusion finale.')
else:
    print('Nouvel entraînement non collecté : aucune métrique finale à comparer pour le moment.')

## 11. Récent seul : conserver vraiment la classe sports ball

Même **1 547 train / 168 validation**, sans TV, mêmes réglages à 960 px. Nous gardons les caractéristiques du classifieur COCO et ses poids de sortie `sports ball` : **classe 32 → 0** dans les deux branches, aux trois échelles. Ce n'est pas la classe COCO person.

Un simple nom identique ne conserve ici que les biais, car le constructeur mono-classe réduit la largeur du classifieur de 80 à 64. Nous conservons la largeur 80 : **+1,19 % de paramètres**, donc ce test porte sur le transfert complet, pas seulement le renommage. Poids et prédictions brutes sont vérifiés avant entraînement ; ils peuvent ensuite évoluer.

La cellule lit uniquement les rapports locaux : aucun calcul payant ni inférence. Après la fin du job : `python scripts/mobile_training_hf.py collect --config configs/mobile_nano_sports_ball_960.json`, puis réexécuter cette cellule. Le nom de classe ne doit pas être confondu avec une garantie de performance sur les gros plans.

In [ ]:
SPORT_CONFIG = json.loads((PROJECT_ROOT / 'configs/mobile_nano_sports_ball_960.json').read_text())
SPORT_JOB_PATH = PROJECT_ROOT / 'models/reports' / (SPORT_CONFIG['experiment_id'] + '_job.json')
if SPORT_JOB_PATH.exists():
    SPORT_JOB = json.loads(SPORT_JOB_PATH.read_text())
    print('Dernier état enregistré (pas en direct) :', SPORT_JOB['status'])
    if SPORT_JOB.get('job_url'):
        display(Markdown('[Job HF sports ball](' + SPORT_JOB['job_url'] + ')'))
    SPORT_CHECK = SPORT_JOB.get('transfer_preflight')
    if SPORT_CHECK:
        print('Classe', SPORT_CHECK['source_class_id'], '→', SPORT_CHECK['target_class_id'])
        print('Poids conservés :', SPORT_CHECK['all_weight_checks_passed'], '; prédictions brutes :', SPORT_CHECK['dense_prediction_check']['passed'])
        print('Paramètres :', SPORT_CHECK['target_parameters'])
SPORT_ARTIFACTS = PROJECT_ROOT / 'models/checkpoints/baseline_runs' / SPORT_CONFIG['experiment_id'] / 'artifacts'
SPORT_REPORT_PATH = SPORT_ARTIFACTS / 'training_report.json'
if SPORT_REPORT_PATH.exists():
    SPORT_REPORT = json.loads(SPORT_REPORT_PATH.read_text())
    if SPORT_REPORT['status'] == 'complete':
        before, after = SPORT_REPORT['validation']['modern_only'], SPORT_REPORT['validation']['mobile']
        lines = ['| Même validation @960 | Ancien récent | Transfert sports ball | Écart |', '|---|---:|---:|---:|']
        for title, key in [('mAP50', 'map50'), ('mAP50–95', 'map50_95')]:
            b, a = before[key], after[key]
            lines.append(f'| {title} | {b:.1%} | {a:.1%} | {(a-b)*100:+.1f} pts |')
        for title, key in [('Précision', 'precision'), ('Rappel', 'recall'), ('F1', 'f1')]:
            b, a = before['fixed'][key], after['fixed'][key]
            lines.append(f'| {title} | {b:.1%} | {a:.1%} | {(a-b)*100:+.1f} pts |')
        for title, key in [('Bonnes détections', 'tp'), ('Fausses détections', 'fp'), ('Ballons ratés', 'fn'), ('FP sur négatifs', 'fp_on_negatives')]:
            b, a = before['fixed'][key], after['fixed'][key]
            lines.append(f'| {title} | {b} | {a} | {a-b:+d} |')
        display(Markdown('\n'.join(lines)))
        for name in ['modern_only', 'mobile']:
            path = SPORT_ARTIFACTS / 'evaluation' / name / 'fixed_confusion_matrix.png'
            if path.exists():
                display(Image(filename=str(path), width=450))
        path = SPORT_ARTIFACTS / 'train/results.png'
        if path.exists():
            display(Image(filename=str(path), width=1000))
    else:
        print('Rapport intermédiaire : pas encore de résultat final.')
else:
    print('Résultats sports ball pas encore collectés.')

## 12. Données enrichies : shorts revus + gros plans personnels

**2 595 images de train** : 1 547 modernes déjà utilisées + 603 shorts vérifiés + 445 gros plans. Les 456 gros plans ont tous été revus ; seules les 11 exclusions sont retirées, sans filtre de confiance. Corrections humaines prioritaires, propositions inchangées acceptées sur ta confirmation, aucun brouillon présent.

Même validation de 168 images, aucun test ajouté au train, aucune image Infactory TV. Tous les gros plans restent dans le train comme une seule session ; il faudra une autre session pour mesurer leur généralisation. Les matchs des shorts ne sont pas tous identifiés : on vérifie les doublons exacts, mais cela n'exclut pas un recouvrement de scènes recadrées.

YOLO26n à **960 px**, COCO avec transfert complet vérifié de sports ball (32 → 0). On repart de COCO ; le meilleur poids sports ball v001 interrompu sert uniquement de référence avant/après. **30 époques maximum, patience 8**, batch 8. HF A10G privé, limite dure 90 minutes, sauvegarde distante toutes les deux époques. Le volume et le programme d'optimisation changent tous les deux.

La cellule suivante consulte le statut et récupère les résultats uniquement si le job est fini : **elle ne lance jamais un entraînement payant**. Les loss, matrices et métriques apparaîtront ensuite ici.

**Résultat récupéré :** 25 époques, arrêt anticipé après huit époques sans amélioration ; meilleur poids à l'époque 17. Environ 19 min 44 s d'entraînement, hors préparation et évaluations. Au seuil 0,25, l'enrichissement trouve sept ballons de plus, mais ajoute 25 fausses détections. F1 : 74,1 % → 70,5 % ; mAP50 : 76,3 % → 73,7 %. Ce n'est pas une amélioration globale sur cette validation, et cela ne tranche pas les performances sur de nouveaux gros plans. Les deux modèles restent conservés.

Les pertes train continuent de baisser tandis que la validation plafonne et fluctue : davantage d'époques ne sont pas justifiées par cet essai. Ce n'est pas, seul, une preuve de surapprentissage. **Attention :** les précision/rappel dans `train/results.png` suivent le point de fonctionnement choisi par Ultralytics, pas nécessairement le seuil fixe 0,25 du tableau ci-dessous.

In [ ]:
from pathlib import Path
import json, subprocess, sys
from IPython.display import display, Markdown, Image
from huggingface_hub import HfApi
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'configs/mobile_nano_enriched_sports_ball_960.json').exists())
ENRICHED_CONFIG_PATH = PROJECT_ROOT / 'configs/mobile_nano_enriched_sports_ball_960.json'
ENRICHED_CONFIG = json.loads(ENRICHED_CONFIG_PATH.read_text())
ENRICHED_DATA = json.loads((PROJECT_ROOT / ENRICHED_CONFIG['dataset'] / 'build_report.json').read_text())
assert ENRICHED_DATA['validation_identical_to_baseline'] and ENRICHED_DATA['test_images_included'] == 0 and ENRICHED_DATA['tv_images_included'] == 0
lines = ['| Split | Images | Avec ballon | Sans ballon |', '|---|---:|---:|---:|']
for split, counts in ENRICHED_DATA['splits'].items():
    lines.append(f"| {split} | {counts['images']} | {counts['positive']} | {counts['negative']} |")
display(Markdown('\n'.join(lines)))
print('Sources :', ENRICHED_DATA['sources'])
ENRICHED_JOB_PATH = PROJECT_ROOT / 'models/reports' / (ENRICHED_CONFIG['experiment_id'] + '_job.json')
if ENRICHED_JOB_PATH.exists():
    ENRICHED_JOB = json.loads(ENRICHED_JOB_PATH.read_text())
    if ENRICHED_JOB.get('job_id'):
        info = HfApi().inspect_job(job_id=ENRICHED_JOB['job_id'])
        print('État actuel :', info.status.stage)
        display(Markdown('[Ouvrir le job HF](' + ENRICHED_JOB['job_url'] + ')'))
        if info.status.stage == 'COMPLETED':
            subprocess.run([sys.executable, str(PROJECT_ROOT / 'scripts/mobile_training_hf.py'), 'collect', '--config', str(ENRICHED_CONFIG_PATH)], cwd=PROJECT_ROOT, check=True)
        elif info.status.message:
            print(info.status.message)
    else:
        print('Aucun job démarré :', ENRICHED_JOB['status'], ENRICHED_JOB.get('message', ''))
else:
    print('Aucun job enregistré ; cette cellule ne le lance pas.')
ENRICHED_ARTIFACTS = PROJECT_ROOT / 'models/checkpoints/baseline_runs' / ENRICHED_CONFIG['experiment_id'] / 'artifacts'

In [ ]:
ENRICHED_REPORT_PATH = ENRICHED_ARTIFACTS / 'training_report.json'
if ENRICHED_REPORT_PATH.exists():
    report = json.loads(ENRICHED_REPORT_PATH.read_text())
    if report['status'] == 'complete':
        before, after = report['validation']['sports_ball_v001'], report['validation']['mobile']
        lines = ['| Même validation @960 | Sports ball v001 | Enrichi v002 | Écart |', '|---|---:|---:|---:|']
        for title, key in [('mAP50', 'map50'), ('mAP50–95', 'map50_95')]:
            b, a = before[key], after[key]
            lines.append(f'| {title} | {b:.1%} | {a:.1%} | {(a-b)*100:+.1f} pts |')
        for title, key in [('Précision', 'precision'), ('Rappel', 'recall'), ('F1', 'f1')]:
            b, a = before['fixed'][key], after['fixed'][key]
            lines.append(f'| {title} | {b:.1%} | {a:.1%} | {(a-b)*100:+.1f} pts |')
        for title, key in [('Bonnes détections', 'tp'), ('Fausses détections', 'fp'), ('Ballons ratés', 'fn'), ('FP sur négatifs', 'fp_on_negatives')]:
            b, a = before['fixed'][key], after['fixed'][key]
            lines.append(f'| {title} | {b} | {a} | {a-b:+d} |')
        display(Markdown('\n'.join(lines)))
        print('Confiance ≥ 0,25 ; association IoU ≥ 0,50. mAP : tous les seuils.')
        print('Époques terminées :', report['epochs_logged'])
        for name in ['sports_ball_v001', 'mobile']:
            path = ENRICHED_ARTIFACTS / 'evaluation' / name / 'fixed_confusion_matrix.png'
            if path.exists():
                display(Image(filename=str(path), width=450))
        for relative in ['train/results.png', 'evaluation/mobile/BoxPR_curve.png', 'evaluation/mobile/BoxF1_curve.png']:
            path = ENRICHED_ARTIFACTS / relative
            if path.exists():
                display(Image(filename=str(path), width=1000))
        print('Poids :', ENRICHED_ARTIFACTS / 'best.pt')
    else:
        print('Rapport intermédiaire : ne pas conclure avant la fin.')
else:
    print('Résultats pas encore collectés. Exécuter la cellule de statut après la fin du job.')

## 13. Corriger les faux positifs proches : négatifs de la maison

Les neuf clips IMG_7016–IMG_7024 font environ 12 secondes. Ils sont confirmés entièrement sans ballon par l'utilisateur ; inspection visuelle des extraits de début/milieu. On extrait **une frame sur six : 63 images**, sans redimensionner, avec labels YOLO explicitement vides. Aucun appel Luna ni classe négative supplémentaire.

**2 658 train**, dont 2 274 positifs et 384 négatifs ; **168 val identiques**. Tous les gros plans et shorts restent inclus. Pas de TV, pas de test ajouté au train. Les neuf nouvelles vidéos sont toutes dans le train, en une session maison. Le diagnostic avant/après sur ces 63 frames mesure donc l'apprentissage, **pas la généralisation** : il faudra d'autres objets/prises pour cela.

On continue depuis le **best.pt enrichi v002**, pour conserver ses acquis sur les ballons proches. Ce n'est ni un retour à COCO ni une reprise de l'ancien optimizer. Les poids et les deux branches du classifieur à 80 canaux sont conservés et vérifiés avant optimisation. 960 px, **15 époques maximum**, patience 5, AdamW **lr0=0,0002** (cinq fois plus petit), batch 8. HF A10G privé, limite dure 45 minutes ; checkpoint distant toutes les deux époques. Le choix du meilleur checkpoint reste basé sur la validation foot.

Ces cellules ne lancent jamais un job payant : elles consultent son état, récupèrent les résultats s'il est fini et présentent les mesures. On surveille à la fois les faux positifs et le rappel des vrais ballons.

**Résultat récupéré :** 14 époques, meilleur poids à l'époque 9, environ 11 min 26 s d'entraînement. Sur les 63 négatifs **de train**, 91 fausses boîtes sur 51 images avant → zéro après. Cela démontre l'apprentissage de ces scènes, pas la généralisation. Sur la validation foot inchangée : F1 70,5 % → 75,3 %, précision 71,5 % → 85,7 %, rappel 69,4 % → 67,2 %. Il retire 22 faux positifs mais manque trois ballons supplémentaires. L'ajout des négatifs et le fine-tuning changent ensemble ; leur contribution respective n'est pas isolée. Vérifier les gros plans et d'autres objets sur de nouvelles prises.

In [ ]:
from pathlib import Path
import json, subprocess, sys
from IPython.display import display, Markdown, Image
from huggingface_hub import HfApi
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'configs/mobile_nano_hard_negatives_960.json').exists())
NEGATIVE_CONFIG_PATH = PROJECT_ROOT / 'configs/mobile_nano_hard_negatives_960.json'
NEGATIVE_CONFIG = json.loads(NEGATIVE_CONFIG_PATH.read_text())
NEGATIVE_DATA = json.loads((PROJECT_ROOT / NEGATIVE_CONFIG['dataset'] / 'build_report.json').read_text())
assert NEGATIVE_DATA['validation_identical_to_baseline'] and NEGATIVE_DATA['test_images_included'] == 0 and NEGATIVE_DATA['tv_images_included'] == 0
lines = ['| Split | Images | Avec ballon | Sans ballon |', '|---|---:|---:|---:|']
for split, counts in NEGATIVE_DATA['splits'].items():
    lines.append(f"| {split} | {counts['images']} | {counts['positive']} | {counts['negative']} |")
display(Markdown('\n'.join(lines)))
print('Initialisation : meilleur poids enrichi v002 ; lr0 =', NEGATIVE_CONFIG['lr0'])
NEGATIVE_JOB_PATH = PROJECT_ROOT / 'models/reports' / (NEGATIVE_CONFIG['experiment_id'] + '_job.json')
if NEGATIVE_JOB_PATH.exists():
    job = json.loads(NEGATIVE_JOB_PATH.read_text())
    if job.get('job_id'):
        info = HfApi().inspect_job(job_id=job['job_id'])
        print('État actuel :', info.status.stage)
        display(Markdown('[Ouvrir le job HF](' + job['job_url'] + ')'))
        if info.status.stage == 'COMPLETED':
            subprocess.run([sys.executable, str(PROJECT_ROOT / 'scripts/mobile_training_hf.py'), 'collect', '--config', str(NEGATIVE_CONFIG_PATH)], cwd=PROJECT_ROOT, check=True)
        elif info.status.message:
            print(info.status.message)
    else:
        print('Aucun job démarré :', job['status'], job.get('message', ''))
else:
    print('Aucun job enregistré ; cette cellule ne le lance pas.')
NEGATIVE_ARTIFACTS = PROJECT_ROOT / 'models/checkpoints/baseline_runs' / NEGATIVE_CONFIG['experiment_id'] / 'artifacts'

In [ ]:
path = NEGATIVE_ARTIFACTS / 'training_report.json'
if path.exists():
    report = json.loads(path.read_text())
    if report['status'] == 'complete':
        before, after = report['validation']['enriched_v002'], report['validation']['mobile']
        lines = ['| Validation foot identique @960 | Enrichi v002 | Négatifs v003 | Écart |', '|---|---:|---:|---:|']
        for title, key in [('mAP50', 'map50'), ('mAP50–95', 'map50_95')]:
            b, a = before[key], after[key]
            lines.append(f'| {title} | {b:.1%} | {a:.1%} | {(a-b)*100:+.1f} pts |')
        for title, key in [('Précision', 'precision'), ('Rappel', 'recall'), ('F1', 'f1')]:
            b, a = before['fixed'][key], after['fixed'][key]
            lines.append(f'| {title} | {b:.1%} | {a:.1%} | {(a-b)*100:+.1f} pts |')
        for title, key in [('Bonnes détections', 'tp'), ('Fausses détections', 'fp'), ('Ballons ratés', 'fn')]:
            b, a = before['fixed'][key], after['fixed'][key]
            lines.append(f'| {title} | {b} | {a} | {a-b:+d} |')
        display(Markdown('\n'.join(lines)))
        print('Confiance ≥ 0,25 ; association IoU ≥ 0,50. mAP : tous les seuils.')
        diagnostic = report.get('negative_training_diagnostic', {})
        if diagnostic:
            b, a = diagnostic['enriched_v002'], diagnostic['mobile']
            assert b['images'] == a['images'] == 63 and not a['independent_generalization_test']
            display(Markdown(f"### Diagnostic sur les 63 négatifs DE TRAIN — pas un test indépendant\n\n| Mesure à conf ≥ 0,25 | Avant | Après |\n|---|---:|---:|\n| Fausses boîtes | {b['fp']} | {a['fp']} |\n| Images avec une fausse boîte | {b['images_with_fp']} | {a['images_with_fp']} |\n| Proportion d'images avec FP | {b['fraction_images_with_fp']:.1%} | {a['fraction_images_with_fp']:.1%} |"))
        print('Époques terminées :', report['epochs_logged'])
        for name in ['enriched_v002', 'mobile']:
            image = NEGATIVE_ARTIFACTS / 'evaluation' / name / 'fixed_confusion_matrix.png'
            if image.exists():
                display(Image(filename=str(image), width=450))
        learning = NEGATIVE_ARTIFACTS / 'train/results.png'
        if learning.exists():
            display(Image(filename=str(learning), width=1000))
        print('Nouveau poids :', NEGATIVE_ARTIFACTS / 'best.pt')
    else:
        print('Rapport intermédiaire : attendre la fin pour conclure.')
else:
    print('Résultats pas encore collectés. Réexécuter la cellule de statut une fois le job fini.')